In [25]:
from pathlib import Path

def get_project_root():
    p = Path.cwd().resolve()
    # Go up until we find a folder that looks like the real repo root
    while p != p.parent:
        if (p / ".git").exists() or (p / ".dvc").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (.git or .dvc not found).")

PROJECT_ROOT = get_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA exists:", (PROJECT_ROOT / "data").exists())


PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
DATA exists: True


In [27]:
from pathlib import Path
from PIL import Image

# ===================== 0) Find PROJECT ROOT =====================
def get_project_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / ".git").exists() or (p / ".dvc").exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found (.git or .dvc missing).")

PROJECT_ROOT = get_project_root()

# ===================== 1) Paths =====================
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "archive (1)" / "data1a"
OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "archive1"

# ===================== 2) Settings =====================
TARGET_SIZE = (224, 224)
classes = ["00-damage", "01-whole"]

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT exists:", RAW_ROOT.exists())
print("RAW_ROOT:", RAW_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("TARGET_SIZE:", TARGET_SIZE)

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"RAW_ROOT not found: {RAW_ROOT}")

# ===================== 3) Create output folders =====================
for c in classes:
    (OUT_ROOT / "train" / c).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "val" / c).mkdir(parents=True, exist_ok=True)

# ===================== 4) Helper: count images =====================
def count_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix in exts)

# ===================== 5) Check RAW counts =====================
train_path = RAW_ROOT / "training"
val_path   = RAW_ROOT / "validation"

print("\n--- RAW COUNTS ---")
for c in classes:
    print(f"TRAIN {c}:", count_images(train_path / c))
for c in classes:
    print(f"VAL   {c}:", count_images(val_path / c))




CWD: c:\Users\User\Desktop\Vehicle_Damage_Detection\notebooks\02_preprocess
PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_ROOT exists: True
RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1
TARGET_SIZE: (224, 224)

--- RAW COUNTS ---
TRAIN 00-damage: 920
TRAIN 01-whole: 920
VAL   00-damage: 230
VAL   01-whole: 230


In [29]:
from pathlib import Path
from PIL import Image

# ================= PROJECT ROOT =================
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD
while PROJECT_ROOT.name != "Vehicle_Damage_Detection":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

# ================= PATHS =================
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "archive (1)" / "data1a"
OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "archive1"

TARGET_SIZE = (224, 224)
classes = ["00-damage", "01-whole"]

train_in = RAW_ROOT / "training"
val_in   = RAW_ROOT / "validation"

print("RAW_ROOT exists:", RAW_ROOT.exists())
print("TRAIN exists:", train_in.exists())
print("VAL exists:", val_in.exists())
print("OUT_ROOT:", OUT_ROOT)

# ================= PREPROCESS FUNCTION =================
def process_split(split_in: Path, split_out: Path):
    exts = {".jpg", ".jpeg", ".png"}
    processed = 0
    skipped = 0

    for c in classes:
        in_dir = split_in / c
        out_dir = split_out / c
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\nProcessing: {in_dir}")

        for img_path in in_dir.iterdir():
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in exts:
                continue

            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(TARGET_SIZE)

                out_path = out_dir / img_path.name
                img.save(out_path, format="JPEG", quality=95)

                processed += 1
            except Exception as e:
                skipped += 1
                print("Skipped:", img_path.name, "|", e)

    return processed, skipped

# ================= RUN =================
print("\nPreprocessing TRAIN...")
train_processed, train_skipped = process_split(train_in, OUT_ROOT / "train")

print("\nPreprocessing VAL...")
val_processed, val_skipped = process_split(val_in, OUT_ROOT / "val")

print("\nDONE")
print("Train processed:", train_processed, "| skipped:", train_skipped)
print("Val processed  :", val_processed,  "| skipped:", val_skipped)
print("Saved to:", OUT_ROOT)


PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_ROOT exists: True
TRAIN exists: True
VAL exists: True
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1

Preprocessing TRAIN...

Processing: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a\training\00-damage

Processing: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a\training\01-whole

Preprocessing VAL...

Processing: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a\validation\00-damage

Processing: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (1)\data1a\validation\01-whole

DONE
Train processed: 1840 | skipped: 0
Val processed  : 460 | skipped: 0
Saved to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1


In [32]:
from pathlib import Path

# Force project root
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD
while PROJECT_ROOT.name != "Vehicle_Damage_Detection":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Correct absolute paths
train_damage = PROJECT_ROOT / "data" / "processed" / "archive1" / "train" / "00-damage"
train_whole  = PROJECT_ROOT / "data" / "processed" / "archive1" / "train" / "01-whole"
val_damage   = PROJECT_ROOT / "data" / "processed" / "archive1" / "val"   / "00-damage"
val_whole    = PROJECT_ROOT / "data" / "processed" / "archive1" / "val"   / "01-whole"

def count_images(path: Path):
    exts = {".jpg", ".jpeg", ".png"}
    return sum(1 for p in path.iterdir() if p.is_file() and p.suffix.lower() in exts)

print("Processed TRAIN damage:", count_images(train_damage))
print("Processed TRAIN whole :", count_images(train_whole))
print("Processed VAL damage  :", count_images(val_damage))
print("Processed VAL whole   :", count_images(val_whole))


Processed TRAIN damage: 920
Processed TRAIN whole : 920
Processed VAL damage  : 230
Processed VAL whole   : 230
